<a href="https://colab.research.google.com/github/vaya75e/ai/blob/main/Lab4ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<-  Thursday 13 Aug 2026 ->
ด่านที่ 1เปิดสมุดพกจริง

In [25]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix
bc = load_breast_cancer()
X = bc.data
y = 1 - bc.target        # กลับข้างให้ 1 แปลว่า เป็นมะเร็ง
#1 = มะเร็ง
#0 = ไม่เป็นมะเร็ง
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)
pred = tree.predict(X_test)

# test_size = 0.3 => เอา 30% ไปสอบ / เหลือ 70% ไว้ฝึก
# Training ≈ 398 คน / Test ≈ 171 คน
# 569 คือจำนวนแถวของ Dataset breast_cancer ที่โหลดมาจาก sklearn ไม่ได้มาจากโค้ด train_test_split โดยตรง
# random_state=42 ทำให้การสุ่มแบ่งข้อมูล เหมือนเดิมทุกครั้งที่รัน ถ้าไม่ใส่ อาจสุ่มได้คนละชุดทุกครั้ง
# stratify=y => มันช่วยให้สัดส่วน มะเร็ง / ไม่มะเร็งใน Train และ Test ใกล้เคียงกับข้อมูลเดิม
# tree.fit(X_train, y_train) คือขั้นตอน Training โดย X_train = ข้อมูลผู้ป่วย / y_train = คำตอบจริง
# pred = tree.predict(X_test) คือ ให้โมเดลสอบ , เอา ข้อสอบที่โมเดลไม่เคยเห็น X_test ให้มันตอบ
# โดย 1 = โมเดลบอกว่าเป็นมะเร็ง 0 = โมเดลบอกว่าไม่เป็นมะเร็ง

print(confusion_matrix(y_test, pred))
# TN = 100 จริง ๆ ไม่เป็นมะเร็ง โมเดลก็บอกไม่เป็น
# FP = 7 จริง ๆ ไม่เป็นมะเร็ง แต่โมเดลบอกเป็นมะเร็ง "แจ้งเตือนผิด"
# FN = 10 จริง ๆ เป็นมะเร็ง แต่โมเดลบอกไม่เป็นมะเร็ง "ปล่อยผู้ป่วยหลุด"
# TP = 54 จริง ๆ เป็นมะเร็ง โมเดลก็บอกเป็นมะเร็ง

[[100   7]
 [ 10  54]]


ด่านที่ 2จับผิด accuracy

In [26]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import recall_score

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

print("accuracy เดามั่ว", dummy.score(X_test, y_test))
print("recall เดามั่ว", recall_score(y_test, dummy.predict(X_test)))

accuracy เดามั่ว 0.6257309941520468
recall เดามั่ว 0.0


In [27]:
# Precision
# TP / (TP+FP)
54 / (54 + 7)

0.8852459016393442

In [28]:
# Recall
# TP / (TP+FN)
54 / (54 + 10)

0.84375

In [29]:
# F1 score
# F1 = 2 × (precision × recall) / (precision + recall)
2 * ( 0.8852459016393442*0.84375) / (0.8852459016393442 + 0.84375)

0.864

ด่านที่ 3คำนวณเข็มวัดด้วยมือ

In [30]:
from sklearn.metrics import classification_report
print(classification_report(y_test, pred,
      target_names=["ไม่เป็นมะเร็ง", "เป็นมะเร็ง"], digits=3))

               precision    recall  f1-score   support

ไม่เป็นมะเร็ง      0.909     0.935     0.922       107
   เป็นมะเร็ง      0.885     0.844     0.864        64

     accuracy                          0.901       171
    macro avg      0.897     0.889     0.893       171
 weighted avg      0.900     0.901     0.900       171



ด่านที่ 4พลังของการปรับสเกล

In [31]:
from sklearn.datasets import load_wine
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

wine = load_wine()
Xw, yw = wine.data, wine.target
Xw_train, Xw_test, yw_train, yw_test = train_test_split(
    Xw, yw, test_size=0.3, random_state=42)

knn_raw = KNeighborsClassifier()
knn_raw.fit(Xw_train, yw_train)
print("ไม่ปรับสเกล", knn_raw.score(Xw_test, yw_test))

knn_scaled = make_pipeline(StandardScaler(), KNeighborsClassifier())
knn_scaled.fit(Xw_train, yw_train)
print("ปรับสเกล", knn_scaled.score(Xw_test, yw_test))

ไม่ปรับสเกล 0.7407407407407407
ปรับสเกล 0.9629629629629629


ด่านที่ 5ต่อท่อกันรั่วด้วย Pipeline

In [32]:
from sklearn.model_selection import cross_val_score

pipe = make_pipeline(StandardScaler(), KNeighborsClassifier())
scores = cross_val_score(pipe, Xw, yw, cv=5)
print("คะแนน 5 รอบ", scores.round(4))
print("เฉลี่ย", scores.mean().round(4))

คะแนน 5 รอบ [0.9444 0.9444 0.9722 1.     0.8857]
เฉลี่ย 0.9494


ด่านที่ 6สร้างข้อมูลสกปรกด้วยมือตัวเอง

In [33]:
import pandas as pd

rng = np.random.default_rng(42)
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df["region"] = rng.choice(["ไร่เหนือ", "ไร่กลาง", "ไร่ใต้"], size=len(df))
df["target"] = wine.target

In [34]:
df["region"]

,region
0,ไร่เหนือ
1,ไร่ใต้
2,ไร่กลาง
3,ไร่กลาง
4,ไร่กลาง
...,...
173,ไร่ใต้
174,ไร่ใต้
175,ไร่กลาง
176,ไร่ใต้


In [35]:
# เจาะค่าให้หาย 4 คอลัมน์ คอลัมน์ละ 15 ช่อง
for col in ["alcohol", "magnesium", "flavanoids", "proline"]:
    idx = rng.choice(len(df), size=15, replace=False)
    df.loc[idx, col] = np.nan

In [36]:
# แอบยัดแถวซ้ำ 10 แถว แล้วสับไพ่
dups = df.sample(10, random_state=42)
df_dirty = pd.concat([df, dups], ignore_index=True)
df_dirty = df_dirty.sample(frac=1, random_state=42).reset_index(drop=True)
df_dirty.to_csv("wine_dirty.csv", index=False)

In [37]:
print("ขนาด", df_dirty.shape)
print("ค่าหายรวม", df_dirty.isna().sum().sum())
print("แถวซ้ำ", df_dirty.duplicated().sum())

ขนาด (188, 15)
ค่าหายรวม 62
แถวซ้ำ 10


ด่านที่ 7สองทางล้างข้อมูล

In [38]:
df_dirty = pd.read_csv("wine_dirty.csv")
X_bad = df_dirty.drop(columns=["region", "target"]).values
y_bad = df_dirty["target"].values

pipe = make_pipeline(StandardScaler(), KNeighborsClassifier())
pipe.fit(X_bad, y_bad)      # บรรทัดนี้จะพัง

ValueError: Input X contains NaN.
KNeighborsClassifier does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [39]:
df_naive = df_dirty.dropna()
print("เหลือแถว", len(df_naive), "จาก", len(df_dirty))

เหลือแถว 137 จาก 188


7.2 ทางประณีตงานครัวสามอย่างจากหนังสือ

In [40]:
# 1) ลบแถวซ้ำ
df_clean = df_dirty.drop_duplicates().reset_index(drop=True)
print("หลังลบแถวซ้ำ", len(df_clean))

หลังลบแถวซ้ำ 178


In [41]:
# 2) แปลงคำเป็นตัวเลขด้วย one-hot
df_clean = pd.get_dummies(df_clean, columns=["region"])

In [42]:
# ผลเฉลยห้ามเอาไปเทรน เพราะมันต้องถูกอยู่แล้ว
df_clean

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target,region_ไร่กลาง,region_ไร่เหนือ,region_ไร่ใต้
0,14.30,1.92,2.72,20.0,NaN,2.80,NaN,0.33,1.97,6.20,1.07,2.65,1280.0,0,True,False,False
1,12.85,3.27,2.58,22.0,106.0,1.65,0.60,0.60,0.96,5.58,0.87,2.11,570.0,2,True,False,False
2,14.19,1.59,2.48,16.5,108.0,3.30,3.93,0.32,1.86,8.70,1.23,2.82,1680.0,0,False,False,True
3,13.63,1.81,2.70,17.2,112.0,2.85,2.91,0.30,1.46,7.30,1.28,2.88,1310.0,0,False,False,True
4,12.37,1.17,1.92,19.6,78.0,2.11,2.00,0.27,1.04,4.68,1.12,3.48,510.0,1,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,13.86,1.51,2.67,25.0,86.0,2.95,NaN,0.21,1.87,3.38,1.36,3.16,410.0,1,False,True,False
174,12.25,1.73,2.12,19.0,80.0,1.65,2.03,0.37,1.63,3.40,1.00,3.17,510.0,1,False,False,True
175,14.38,1.87,2.38,12.0,102.0,3.30,NaN,0.29,2.96,7.50,1.20,3.00,NaN,0,False,False,True
176,12.69,1.53,2.26,20.7,80.0,1.38,1.46,0.58,1.62,3.05,0.96,2.06,495.0,1,False,False,True


In [43]:
Xc = df_clean.drop(columns=["target"])

In [44]:
yc = df_clean["target"]

In [45]:
Xc_train, Xc_test, yc_train, yc_test = train_test_split( Xc, yc, test_size=0.3, random_state=42)

In [46]:
med = Xc_train.median(numeric_only=True)

In [48]:
# จัดการ missing data/missing column
pipe = make_pipeline(StandardScaler(), KNeighborsClassifier())
pipe.fit(Xc_train, yc_train)
print("คะแนนกองสอบ", round(pipe.score(Xc_test, yc_test), 4))

คะแนนกองสอบ 0.9259
